# Floor computation from decoding failure rates for $(t,u)$-almost near-codewords

Implementation of Eq. (7) from the paper. The values of $\varepsilon(t,u)$ are read from outputs files, formatted as

$\texttt{FLOOR}\_t\_r\_v\_\texttt{seed}\_\texttt{dec}\_\texttt{type.txt}$

For instance, using 

- $t = 134$
- $r = 12323$
- $v = 71$
- $\texttt{seed} = 0$
- $\texttt{dec}\_\texttt{type} = \texttt{max}$

we consider BIKE parameters for NIST category 1, decoded through $\textsf{BF}\text{-}\textsf{Max}$, where the randomness for the simulation is generated starting from $\texttt{seed} = 0$.

The probability $f(u)$ is computed as described in Appendix B.

##### Computation of $f(u)$

In [1]:
def compute_f(t_i, u_i, v, r):
    RR = Reals(4096)
    pr = RR(0)
    right_term = sum([RR(binomial(t_i,x))*RR(binomial(r-t_i,v-x))/RR(binomial(r,v)) for x in range(u_i)])
    left_term = right_term + RR(binomial(t_i,u_i))*RR(binomial(r-t_i,v-u_i))/RR(binomial(r,v))

    return left_term**r - right_term**r

def compute_pr_theoretical(v,r,t,u):
    RR = Reals(4096)
    pr = RR(0)
    for t1 in range(t+1):
        t2 = t-t1
        pr_t1_t2 = binomial(r,t1)*binomial(r,t2)/binomial(2*r,t)
        term = compute_f(t-t1,u,v,r)*sum([compute_f(t1,x,v,r) for x in range(u)])
        term += compute_f(t1,u,v,r)*compute_f(t-t1,u,v,r)
        term += compute_f(t1,u,v,r)*sum([compute_f(t-t1,x,v,r) for x in range(u)])
        #print("-----> u = ",u,", term = ",round(term,5))
        pr += (pr_t1_t2*term)
    
    return pr

##### Useful functions

In [2]:
def read_dfr_from_file(file_name):
    '''
    Read DFR values from simulation output files
    '''
    f = open(file_name)
    vals_str = f.read()
    dfr_vals = []
    lines =  vals_str.split("\n")
    for i in range(len(lines)-1):
        a = lines[i].split()
        dfr_vals.append([int(a[0]), float(a[1]), float(a[2])])

    return dfr_vals

##### Compute DFR approximation

Read $\varepsilon(t,u)$ from file, then compute the DFR approximation

In [3]:
#Code and sim parameters
t = 134
r = 12323
v = 71
seed = 0
dec_type = "newbike"

file_name = dec_type+"/FLOOR_"+str(t)+"_"+str(r)+"_"+str(v)+"_"+str(seed)+"_"+dec_type+".txt"

dfr_vals = read_dfr_from_file(file_name)

print("DFR values have been read")

DFR values have been read


Compute DFR approximation. Data are formatted as Latex table

In [17]:
dfr_std = 0
dfr_impr = 0

for val in dfr_vals:
    
    u = val[0]
    eps_std = val[1] #epsilon(t,u) for standard decoder
    eps_impr = val[2] #epsilon(t,u) for improved decoder

    #Compute f(u)
    pr_u = compute_pr_theoretical(v, r, t, u)

    #Standard DFR
    term_std = pr_u*eps_std
    dfr_std += term_std

    #Improved DFR
    term_impr = pr_u*eps_impr
    dfr_impr += term_impr

    #Print data for latex
    print("$"+str(u)+"$ & $2^{"+str(round(log(pr_u,2),2))+"}$ ", end = '')
    if eps_std == 0:
        print("& $"+str(0)+"$ ", end = '')
    else:
        if eps_std == 1:
            print("& $"+str(1)+"$ ", end = '')
        else:
            print("& $2^{"+str(round(log(eps_std,2),2))+"}$ ", end = '')

    if eps_impr == 0:
        print("& $"+str(0)+"$ ", end = '')
    else:
        if eps_impr == 1:
            print("& $"+str(1)+"$ ", end = '')
        else:
            print("& $2^{"+str(round(log(eps_impr,2),2))+"}$ ", end = '')

    print(" & ", end='')
    
    if term_std == 0:
        print("& $"+str(0)+"$ ", end = '')
    else:
        print("& $2^{"+str(round(log(term_std,2),2))+"}$ ", end = '')

    if term_impr == 0:
        print("& $"+str(0)+"$ \\\\")
    else:
        print("& $2^{"+str(round(log(term_impr,2),2))+"}$ \\\\")
    
print(" ")
print("DFR STD = 2^"+str(round(log(dfr_std,2),2)))
print("DFR IMPR = 2^"+str(round(log(dfr_impr,2),2)))

$45$ & $2^{-272.17}$ & $1$ & $0$  & & $2^{-272.17}$ & $0$ \\
$44$ & $2^{-263.35}$ & $1$ & $0$  & & $2^{-263.35}$ & $0$ \\
$43$ & $2^{-254.62}$ & $1$ & $2^{-19.93}$  & & $2^{-254.62}$ & $2^{-274.56}$ \\
$42$ & $2^{-246.0}$ & $2^{-0.0}$ & $2^{-15.23}$  & & $2^{-246.0}$ & $2^{-261.23}$ \\
$41$ & $2^{-237.47}$ & $2^{-0.0}$ & $2^{-11.82}$  & & $2^{-237.47}$ & $2^{-249.3}$ \\
$40$ & $2^{-229.05}$ & $2^{-0.0}$ & $2^{-8.3}$  & & $2^{-229.05}$ & $2^{-237.35}$ \\
$39$ & $2^{-220.71}$ & $2^{-0.02}$ & $2^{-6.31}$  & & $2^{-220.74}$ & $2^{-227.03}$ \\
$38$ & $2^{-212.48}$ & $2^{-0.1}$ & $2^{-4.92}$  & & $2^{-212.58}$ & $2^{-217.4}$ \\
$37$ & $2^{-204.34}$ & $2^{-0.38}$ & $2^{-3.77}$  & & $2^{-204.72}$ & $2^{-208.11}$ \\
$36$ & $2^{-196.3}$ & $2^{-0.9}$ & $2^{-3.29}$  & & $2^{-197.2}$ & $2^{-199.59}$ \\
$35$ & $2^{-188.35}$ & $2^{-1.97}$ & $2^{-4.04}$  & & $2^{-190.32}$ & $2^{-192.39}$ \\
$34$ & $2^{-180.49}$ & $2^{-3.61}$ & $2^{-4.9}$  & & $2^{-184.1}$ & $2^{-185.39}$ \\
$33$ & $2^{-172.73}$ & $2^{

### Print DFR approximation for several values of $t$

In [31]:
#Parameters
r = 2003
v = 13
seed = 0
dec_type = "out"

t_values = range(40, 115, 5)

dfr_vals_std = []
dfr_vals_impr = []

for t in t_values:
    
    print("Doing t = "+str(t), end = ' ')
    
    #Read
    file_name = dec_type+"/FLOOR_"+str(t)+"_"+str(r)+"_"+str(v)+"_"+str(seed)+"_"+dec_type+".txt"
    dfr_vals = read_dfr_from_file(file_name)

    #Compute DFR values
    dfr_std = 0
    dfr_impr = 0
    for val in dfr_vals:
        u = val[0]
        eps_std = val[1]
        eps_impr = val[2]   
        
        pr_u = compute_pr_theoretical(v, r, t, u)
        
        dfr_std += (pr_u*eps_std)
        dfr_impr += (pr_u*eps_impr)

    #Append computed DFR values
    dfr_vals_std.append((t, dfr_std))
    dfr_vals_impr.append((t, dfr_impr)) 

    print(" ---> DONE")

Doing t = 40  ---> DONE
Doing t = 45  ---> DONE
Doing t = 50  ---> DONE
Doing t = 55  ---> DONE
Doing t = 60  ---> DONE
Doing t = 65  ---> DONE
Doing t = 70  ---> DONE
Doing t = 75  ---> DONE
Doing t = 80  ---> DONE
Doing t = 85  ---> DONE
Doing t = 90  ---> DONE
Doing t = 95  ---> DONE
Doing t = 100  ---> DONE
Doing t = 105  ---> DONE
Doing t = 110  ---> DONE


Print values for TIKZ figures

In [33]:
print("DFR STANDARD")
for vals in dfr_vals_std:
    t = vals[0]
    dfr = vals[1]
    print("("+str(t)+", "+"{:e}".format(dfr)+")")
print(" ")
          
print("DFR IMPROVED")
for vals in dfr_vals_impr:
    t = vals[0]
    dfr = vals[1]
    print("("+str(t)+", "+"{:e}".format(dfr)+")")

DFR STANDARD
(40, 1.201229e-07)
(45, 3.032948e-07)
(50, 7.946899e-07)
(55, 1.627893e-06)
(60, 3.934103e-06)
(65, 9.287999e-06)
(70, 2.189916e-05)
(75, 4.803353e-05)
(80, 1.247567e-04)
(85, 3.115378e-04)
(90, 1.191086e-03)
(95, 1.843822e-02)
(100, 1.264283e-01)
(105, 5.005515e-01)
(110, 8.513660e-01)
 
DFR IMPROVED
(40, 1.988234e-08)
(45, 4.813236e-08)
(50, 1.478289e-07)
(55, 2.655615e-07)
(60, 5.117477e-07)
(65, 1.271827e-06)
(70, 2.295176e-06)
(75, 4.843243e-06)
(80, 1.088362e-05)
(85, 3.432477e-05)
(90, 5.638362e-04)
(95, 1.687015e-02)
(100, 1.245002e-01)
(105, 4.958283e-01)
(110, 8.513632e-01)
